# 02 - Multi-Condition Classification, Optuna Tuning & Probability Calibration
Comprehensive model comparison across Logistic Regression, KNN, SVM, Random Forest, XGBoost, LightGBM, CatBoost, and Stacking Ensemble, followed by Optuna hyperparameter optimization, Stratified K-Fold CV, probability calibration (`CalibratedClassifierCV`), and recall-prioritized threshold tuning.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from src.ml.datasets import generate_synthetic_clinical_cohort
from src.ml.preprocessor import build_preprocessor_pipeline, NUMERIC_FEATURES, CATEGORICAL_FEATURES
from src.ml.classifiers import (
    get_base_classifier, build_stacking_ensemble, evaluate_classifier_cv,     tune_xgboost_optuna, find_recall_prioritized_threshold, CalibratedConditionClassifier
)


### 1. Evaluate Multi-Algorithm Baseline on Diabetes Target


In [ ]:
df = generate_synthetic_clinical_cohort(n_samples=1500, random_state=42)
preprocessor = build_preprocessor_pipeline()
X = preprocessor.fit_transform(df[NUMERIC_FEATURES + CATEGORICAL_FEATURES])
y = df['target_diabetes'].values

algorithms = ['logistic_regression', 'knn', 'svm', 'random_forest', 'xgboost', 'lightgbm', 'catboost']
results = {}
for algo in algorithms:
    clf = get_base_classifier(algo)
    scores = evaluate_classifier_cv(clf, X, y, n_splits=3)
    results[algo] = scores
pd.DataFrame(results).T


### 2. Optuna Hyperparameter Optimization on XGBoost


In [ ]:
best_params = tune_xgboost_optuna(X, y, n_trials=8)
print('Optuna Best Params:', best_params)


### 3. Probability Calibration & Recall-Prioritized Thresholding


In [ ]:
calibrated_clf = CalibratedConditionClassifier('diabetes', base_model_name='random_forest', min_recall=0.92)
calibrated_clf.fit(X, y)
print('Calibrated Decision Threshold (Recall Prioritized):', calibrated_clf.decision_threshold)
print('Evaluation Metrics at Threshold:\n', calibrated_clf.metrics)
